**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment3/'
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# This 下载s COCO 数据集 到 your Drive if it doesn't 已经存在
# (你应该 already have 这个 数据集 来自 a previous notebook!)
# Uncomment following if you don't have it.
# %cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/数据集s/
# !bash get_coco_captioning.sh
# %cd /content/drive/My\ Drive/$FOLDERNAME

In [ ]:
# Some 使用ful python libraries
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git
! pip install decord

# 最先进的预训练图像模型

在前一个练习中，你学习了 [SimCLR](https://arxiv.org/abs/2002.05709)，以及如何使用对比式自监督学习来学习有意义的图像表示。在这个 notebook 中，我们会探索另外两个较新的模型；它们同样旨在学习高质量视觉表示，并且在多种下游任务上展示了强大且稳健的性能。

首先，我们会考察 [CLIP](https://github.com/openai/CLIP) 模型。和 SimCLR 一样，CLIP 使用对比学习目标；但它不是对比同一图像的两个增强视图，而是对比两种不同模态：文本和图像。为了训练 CLIP，OpenAI 从互联网上收集了约 4 亿个图像-文本对，包括 Wikipedia 和图像 alt text 等来源。最终模型学到了丰富的高层图像特征，并在许多视觉 benchmark 上取得了令人印象深刻的 zero-shot 性能。

接下来，我们会探索 [DINO](https://github.com/facebookresearch/dino)。它是一种用于视觉任务的自监督学习方法，在 self-distillation 框架和 multi-crop augmentation 策略中应用对比学习。作者表明，DINO ViT 学到的特征细粒度且语义丰富，并且显式包含图像语义分割相关信息。

# CLIP

如上所述，CLIP 的训练目标同时包含文本和图像，并建立在对比学习原则之上。回顾 SimCLR notebook 中的一句话：
> 对比损失的目标是最大化最终向量 **$z_i = g(h_i)$** 和 **$z_j = g(h_j)$** 之间的一致性。

类似地，CLIP 也被训练来最大化两个向量之间的一致性。不过，由于这些向量来自不同模态，CLIP 使用两个独立编码器：基于 Transformer 的 Text Encoder，以及基于 Vision Transformer（ViT）的 Image Encoder。注意，一些更小、更高效的 CLIP 版本会使用 ResNet 作为 Image Encoder，而不是 ViT。

运行下面的单元，可视化 CLIP 的训练和推理流程。

在预训练阶段，每个 batch 包含多张图像及其对应 caption。每张图像都会被 Image Encoder 独立处理，通常是 Vision Transformer（ViT）或 Convolutional Neural Network（ConvNet）这样的视觉模型，输出图像 embedding $I_n$。同样，每条 caption 会被 Text Encoder 独立处理，生成对应的文本 embedding $T_n$。接下来，我们计算所有图像-文本组合之间的成对相似度，也就是每张图像都与每条 caption 比较，反之亦然。训练目标是最大化所得相似度矩阵对角线上的分数，也就是匹配图像-caption 对 $(I_n, T_n)$ 的分数。通过反向传播，模型会学习给真实匹配分配比错误匹配更高的相似度。

通过这种设置，CLIP 能有效地把图像和文本表示到共享潜在空间中。在这个空间里，语义概念以一种与模态无关的方式编码，从而支持视觉输入和文本输入之间有意义的跨模态比较。

In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/CLIP.png')

**内联问题 1** -

为什么 CLIP 的学习依赖 batch size？如果 batch size 固定，我们可以使用什么策略来学习丰富的图像特征？

$\color{blue}{\textit 你的回答：}$

# 加载 COCO 数据集

我们会使用你训练 RNN 图像描述模型时用过的同一个 captioning 数据集；但这次不是生成 caption，而是看看能否把每张图像匹配到正确的 caption。

In [ ]:
import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import clip
import torch
from tqdm.auto import tqdm

from PIL import Image
from cs231n.clip_dino import *

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-10, np.abs(x) + np.abs(y))))


In [ ]:
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

In [ ]:
# 从磁盘加载 COCO 数据到字典中。
# 这个 is same 数据集 you 使用 用于 RNN captioning notebook :)
data = load_coco_data(pca_features=True)

# 打印数据字典中的所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

In [ ]:
# we're just 使用 loaded captions 来自 COCO, so 我们需要 到 decode them 并 get rid 的 special tokens.
decoded_captions= []
for caption in data['val_captions']:
  caption = decode_captions(caption, data['idx_to_word'])\
    .replace('<START>', '')\
    .replace('<END>', '')\
    .replace('<UNK>', '')\
    .strip()
  decoded_captions.append(caption)

In [ ]:
# lets get 10 样本
mask = np.array([135428, 122586, 122814, 133173, 176639, 163828,  98169,   6931,
        19488, 175760])
first_captions = [decoded_captions[elem] for elem in mask]

img_idxs = data['val_image_idxs'][mask]       # images captions refer to
first_images   = [image_from_url(data['val_urls'][j]) for j in img_idxs]

In [ ]:
for i, (caption, image) in enumerate(zip(first_captions, first_images)):
    plt.imshow(image)
    plt.axis('off')
    caption_str = caption
    plt.title(caption_str)
    plt.show()

# 运行 CLIP 模型

首先，我们会使用预训练 CLIP 模型分别从文本和图像中提取特征。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
# 你可以 check 模型 层 by printing 模型.
# CLIP's 模型 code is available at https://github.com/openai/CLIP/tree/main/clip
# print(clip_模型)

In [ ]:
# First, we encode captions 到 vectors 在 shared embedding space.
# Since we're 使用 a Transformer as text encoder, 我们需要 到 tokenize text first.
text_tokens = clip.tokenize(first_captions).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)

# Sanity check, print 形状
print(text_features.shape)

# 注意： For your 实现s, 使用 above clip_模型.encode_text() 函数. Avoid 使用 clip_模型.前向().

In [ ]:
# Then, we encode images 到 same embedding space.
processed_images = [
    clip_preprocess(Image.fromarray(img)).unsqueeze(0)
    for img in first_images
]
images_tensor = torch.cat(processed_images, dim=0).to(device)

with torch.no_grad():
    image_features = clip_model.encode_image(images_tensor)

# 合理性检查, print 形状
print(image_features.shape)

# 注意： For your 实现s, 使用 above clip_模型.encode_image() 函数. Avoid 使用 clip_模型.前向().

打开 `cs231n/clip_dino.py`，实现 `get_similarity_no_loop`，用于计算文本特征和图像特征之间的相似度分数。运行下面的代码测试你的实现；你应该看到小于 `1e-5` 的相对误差。

In [ ]:
from cs231n.clip_dino import get_similarity_no_loop
torch.manual_seed(231)
np.random.seed(231)
M, N, D = 5, 6, 10

test_text_features = torch.randn(N, D)
test_image_features = torch.randn(M, D)
out = get_similarity_no_loop(test_text_features, test_image_features)

expected_out = np.array([
    [ 0.1867811 , -0.23494351,  0.44155994, -0.18950461,  0.00100103],
    [ 0.17905031, -0.25469488, -0.64330417,  0.25097957, -0.09327742],
    [-0.4407011 , -0.4365381 ,  0.32857686, -0.3765278 ,  0.01049389],
    [ 0.24815483,  0.42157224, -0.08459304,  0.14132318, -0.26935193],
    [ 0.02309848, -0.01441101,  0.5469337 ,  0.6018773 ,  0.21581158],
    [ 0.41579214, -0.014449  , -0.7242257 ,  0.39348006,  0.0822239 ],
]).astype(np.float32)

print("relative error: ", rel_error(out.numpy(), expected_out))

In [ ]:
# Let's 可视化 similarities between our batch 的 images 并 their captions.

similarities = get_similarity_no_loop(text_features, image_features).cpu().detach().numpy()

plt.figure(figsize=(20, 14))
plt.imshow(similarities, vmin=0.1, vmax=0.3)
plt.yticks(range(len(text_features)), first_captions, fontsize=18)
plt.xticks([])
for i, image in enumerate(first_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")
for x in range(similarities.shape[1]):
    for y in range(similarities.shape[0]):
        plt.text(x, y, f"{similarities[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, len(image_features) - 0.5])
plt.ylim([len(text_features) + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)
plt.show()

# Zero Shot 分类器

你会在上面看到匹配的图像-caption 对具有较高相似度。我们可以利用这个性质设计一个不需要任何标注数据的图像分类器，也就是 zero-shot classifier。每个类别可以用合适的自然语言描述来表示，任意输入图像都会被分类到 CLIP embedding 空间中与该图像相似度最高的描述所对应的类别。

在 `cs231n/clip_dino.py` 中实现 `clip_zero_shot_classifier`，并在下面测试。你应该能看到如下预测：

['a person', 'an animal', 'an animal', 'food', 'a person', 'a landscape', 'other', 'other', 'other', 'a person']

In [ ]:
from cs231n.clip_dino import clip_zero_shot_classifier

classes = ["a person", "an animal", "food", "a landscape", "other"]

pred_classes = clip_zero_shot_classifier(
    clip_model, clip_preprocess, first_images, classes, device)

print(pred_classes)

运行下面的单元可视化预测结果。可以看到，CLIP 提供了一种直接的方法，可以在任意类别体系上进行相当合理的 zero-shot 分类。

CLIP 是第一个在不使用任何 ImageNet 图像或标签的情况下，在 ImageNet 分类上超过标准监督训练的模型。原始 CLIP 论文中还有许多类似的有趣实验和分析。

In [ ]:
# 可视化 zero shot 预测
for i, (pred_class, image) in enumerate(zip(pred_classes, first_images)):
    plt.imshow(image)
    plt.axis('off')
    plt.title(pred_class)
    plt.show()

# 使用 CLIP 做图像检索

正如我们使用 CLIP 为每张图像检索匹配类别名一样，也可以使用它根据文本输入检索匹配图像，也就是 semantic image retrieval。在 `cs231n/clip_dino.py` 中实现 `CLIPImageRetriever`，并运行下面两个单元进行测试。每个查询期望的 top 2 输出已在注释中给出。

In [ ]:
from cs231n.clip_dino import CLIPImageRetriever
clip_retriever = CLIPImageRetriever(clip_model, clip_preprocess, first_images, device)

In [ ]:
query = "sports"  # tennis, skateboard
# query = "black 并 white"  # bathroom, zerbas
img_indices = clip_retriever.retrieve(query)

for img_index in img_indices:
    plt.imshow(first_images[img_index])
    plt.axis('off')
    plt.show()

**内联问题 2** -

CLIP 使用对比损失在共享潜在空间中对齐图像和文本表示。你会如何把这个思路扩展到两个以上的模态？

$\color{blue}{\textit 你的回答：}$

# DINO

如前所述，使用普通对比学习方法训练的模型（例如 SimCLR 和 CLIP）需要非常大的 batch size。这使它们计算代价高，并限制了可访问性。后续工作如 [BYOL](https://arxiv.org/abs/2006.07733) 提出了一种替代方法：用 student-teacher 框架避免大量负样本需求。这种方法效果出人意料地好，后来被 [DINO](https://arxiv.org/abs/2104.14294) 采用。

和 SimCLR 类似，DINO 被训练来最大化同一图像不同视图产生的两个向量之间的一致性。不过不同于 SimCLR，DINO 使用两个独立编码器，并且它们的训练方式不同。student network 通过反向传播更新，使其输出匹配 teacher network 的输出。teacher network 不通过反向传播更新；它的权重使用 student 权重的指数滑动平均（EMA）更新。这意味着 teacher 模型变化更慢，可以为 student 提供稳定的学习目标。

运行下面的单元可视化 DINO 的训练流程。

In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/dino.gif')

In [ ]:
# first let's get rid 的 CLIP 模型 该's currently 使用 memory
del clip_model
# Uncomment following if you are 使用 GPU runtime
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [ ]:
# Load sm所有est dino 模型. ViT-S/8. Here ViT-S has ~22M 参数 and
# works on 8x8 patches.
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
dino_model.eval().to(device)


In [ ]:
# image 我们将 be playing 约为 使用
sample_image = Image.fromarray(first_images[0]).convert("RGB")
sample_image

# DINO Attention Maps

由于加载的 DINO checkpoint 基于 ViT 架构，我们可以可视化每个 attention head 关注的内容。下面的代码会生成热力图，显示最终层中各个 head 的 [CLS] token 关注原始图像中的哪些 patch。虽然这个模型用自监督目标训练，并没有显式要求识别图像中的 “结构”，但是仍然会出现一些模式。

你注意到了什么规律吗？

In [ ]:
# Preprocess
from torchvision import transforms as T
transform = T.Compose([
    T.Resize((480, 480)),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
img_tensor = transform(sample_image)
w, h = img_tensor.shape[1:]
img_tensor = img_tensor[None].to(device)

# Extract attention
with torch.no_grad():
    attn = dino_model.get_last_selfattention(img_tensor)[0, :, 0, 1:]
nh, tokens = attn.shape
w_feat, h_feat = w // 8, h // 8
attn = attn.reshape(nh, w_feat, h_feat)
attn = torch.nn.functional.interpolate(attn.unsqueeze(0), scale_factor=8, mode="nearest")[0].cpu().numpy()

# Plot attention heads
fig, axes = plt.subplots(1, nh, figsize=(3 * nh, 3))
for i in range(nh):
    ax = axes[i] if nh > 1 else axes
    ax.imshow(attn[i], cmap='inferno')
    ax.axis('off')
plt.show()

In [ ]:
# Extract patch token 特征 并 discard [CLS] token.
with torch.no_grad():
    all_tokens = dino_model.get_intermediate_layers(img_tensor, n=1)[0]  # (1, 1+N, D)
    patch_tokens = all_tokens[:, 1:, :]  # (N, D)

print(img_tensor.shape)
print(all_tokens.shape)
print(patch_tokens.shape)

**内联问题 3**

上面打印出的 tensor shape 是如何得到的？请解释你的答案。

$\color{blue}{\textit 你的回答：}$

# DINO 特征

为了理解模型在每个 patch 中编码了什么，我们可以可视化每个 patch token 的内容。由于这些 embedding 维度很高，难以直接解释，我们会使用 PCA 找到特征空间中方差最大的方向。

在下一个单元中，我们可视化特征空间中的三个主方差方向。这会揭示 patch embedding 捕捉到的主导结构。

In [ ]:
from sklearn.decomposition import PCA

np.random.seed(231)

# PCA
pca = PCA(n_components=3)
patch_pca = pca.fit_transform(patch_tokens.cpu().numpy()[0])

# Normalize PCA components 到 [0, 1] 用于 RGB display
patch_rgb = (patch_pca - patch_pca.min(0)) / (patch_pca.max(0) - patch_pca.min(0))

# reshape 到 image grid (60x60, 3)
patch_rgb_img = patch_rgb.reshape(60, 60, 3)

# Show as image
plt.figure(figsize=(6, 6))
plt.imshow(patch_rgb_img)
plt.axis('off')
plt.title("Patch Embeddings (PCA → RGB)")
plt.show()

**内联问题 4** -

你在上面的可视化中看到了什么样的结构？如果某个区域持续呈现特定颜色，这意味着什么？如果两个区域颜色明显不同，又意味着什么？请记住，PCA 揭示的是所有 patch 的特征空间中方差最大的方向。一个 patch 的颜色反映了它不同的特征内容。

$\color{blue}{\textit 你的回答：}$

# 基于 DINO 特征的简单分割模型

在上一节中，我们看到 DINO 特征可以提供出人意料地好的分割线索。现在，让我们在 [DAVIS dataset](https://davischallenge.org) 上训练一个简单分割模型来验证这个想法。DAVIS 数据集（Densely Annotated VIdeo Segmentation）是为视频目标分割任务创建的。它为视频中的目标提供逐帧、像素级标注。在这个实验中，我们只使用一个视频中单帧的标注来训练模型，并观察它在同一视频剩余帧上的表现。

我们的模型会刻意保持最小化：我们会按 patch 提取 DINO 特征，并只使用那一帧有标注图像中的 patch，训练一个轻量级 per-patch 分类器。通常情况下，你会在完整数据集上训练，并在包含不同视频的独立验证集上评估。但在这里，我们会测试 DINO 特征的 one-shot 能力。

In [ ]:
from cs231n.clip_dino import DavisDataset

# A helper 类别 到 work 使用 DAVIS 数据集.
# It may take ~5 minutes on first run 的 这个 cell 到 下载 数据集.
davis_ds = DavisDataset()

# Get a specific 测试 video. Do NOT change 这个 用于 submission.
frames, masks = davis_ds.get_sample(7)
num_classes = masks.max() + 1

print(frames.shape, masks.shape, num_classes)

In [ ]:
# Get DINO patch 特征 并 corresponding 类别 标签 用于 a middle frame
train_fi = 40
X_train = davis_ds.process_frames(frames[train_fi:train_fi+1], dino_model, device)[0]
Y_train = davis_ds.process_masks(masks[train_fi:train_fi+1], device)[0]
print(X_train.shape, Y_train.shape)

完成 `cs231n/clip_dino.py` 中 `DINOSegmentation` 类的实现，并运行下面两个单元进行测试。你应该在第一个测试帧上达到大于 0.45 的 mean IoU，在最后一个测试帧上达到大于 0.50 的 mean IoU。为了避免在训练 patch 特征上过拟合，可以考虑设计一个非常轻量的模型，例如线性层或两层 MLP，并使用合适的 weight decay。

你可以使用 GPU runtime 加速训练和评估。如果切换 runtime type，请确保重新运行整个 notebook。

In [ ]:
from cs231n.clip_dino import DINOSegmentation, compute_iou
torch.manual_seed(231)
np.random.seed(231)
dino_segmentation = DINOSegmentation(device, num_classes)
dino_segmentation.train(X_train, Y_train, num_iters=500)


# Test on first, middle, 并 last frame
ious = []
test_fis = [0, train_fi, 98]
gt_viz = []
pred_viz = []
for fi in test_fis:
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.concatenate(gt_viz, 1)
pred_viz = np.concatenate(pred_viz, 1)

In [ ]:
print(f"Mean IoU on first test frames: {ious[0]:.3f}")  # 应为 >0.45
print(f"Mean IoU on last test frames: {ious[2]:.3f}")  # 应为 >0.50

现在可视化结果。运行下面两个单元，显示第一帧、中间帧和最后一帧的 ground truth 与预测分割 mask。注意，中间帧属于训练集，而其他帧是未见过的。

In [ ]:
Image.fromarray(gt_viz)

In [ ]:
Image.fromarray(pred_viz)

现在运行下面三个单元，对整个视频进行评估和可视化。你应该达到大于 0.55 的 mean IoU。保存的可视化视频在 Google Drive 中处理可能需要一些时间，但你可以把它下载到电脑上本地查看。

In [ ]:
# Run on 所有 frames
ious = []
gt_viz = []
pred_viz = []
for fi in range(len(frames)):
  if fi % 20 == 0:
    print(f"{fi} / {len(frames)}")
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.stack(gt_viz)  # T x H x W x 3
pred_viz = np.stack(pred_viz)  # T x H x W x 3
final_viz = np.concatenate([gt_viz, pred_viz], -2)  # T x H x 2W x 3

In [ ]:
print(f"Mean IoU on all frames: {sum(ious) / len(ious):.3f}")  # 应为 >0.55


In [ ]:
def write_video_from_array(array, output_path, fps = 12):
    T, H, W, _ = array.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))
    for i in range(T):
        frame = array[i]
        out.write(frame)
    out.release()
    print(f"Video saved to {output_path}")


# It 可能 take a while 到 process 在 google drive but 你可以 just 下载 it 并 watch on your 计算r
write_video_from_array(final_viz, f"/content/drive/My Drive/{FOLDERNAME}/dino_res.mp4")

**内联问题 5** -

如果你在 CLIP ViT 的 patch 特征上训练分割模型，你预期它会比 DINO 表现更好还是更差？为什么应该是这样？

$\color{blue}{\textit 你的回答：}$